# Summary

Test calling a Vertex AI Search app using a REST API call

In [2]:
import os, sys
import requests
import json
import google.auth
from google.auth.transport.requests import Request


In [3]:
# Set environment variables
utils_path = "utils/"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
api_configs = ApiAuthentication(client="CCC")


In [4]:

class readVaiSearchResults:
    '''
    Class to read and parse Vertex AI Search App results

    '''

    def __init__(self, query: str):
        '''
        Initialize class
        '''

        # Define the endpoint
        self.url = ("https://discoveryengine.googleapis.com/v1alpha/"
                    "projects/1062597788108/locations/global/collections/default_collection/"
                    "engines/web-text-data-store-search_1750123185671/servingConfigs/default_search:search"
                    )


        # Users's query
        self.query = query
        self.pagesize = 10

        # Get credentials
        self.get_gcp_credentials()

        # Call the API
        self.call_api()

        # Parse the response
        self.parse_response()

    def get_gcp_credentials(self):
        '''
        Get GCP credentials need to call the API
        '''

        # Get application default credentials
        credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
        credentials.refresh(Request())

        # Prepare headers
        self.headers = {
            "Authorization": f"Bearer {credentials.token}",
            "Content-Type": "application/json",
            "X-Goog-User-Project": os.environ["GOOGLE_CLOUD_PROJECT"]
        }

    def call_api(self):
        '''
        Call the API to get search results for user's query
        '''

        # Define the request payload
        payload = {
            "query": self.query,
            "pageSize": self.pagesize,
            "queryExpansionSpec": {
                "condition": "AUTO"
            },
            "spellCorrectionSpec": {
                "mode": "AUTO"
            },
            "languageCode": "en-US",
            "userInfo": {
                "timeZone": "America/Los_Angeles"
            }
        }

        # Make the POST request
        self.response = requests.post(self.url, headers=self.headers, data=json.dumps(payload))

        self.response.status_code = self.response.status_code
        self.response_content = self.response.json()

    def parse_response(self):
        '''
        Method to parse response into the elements of interest
        '''


        self.organizations = []
        self.uris = []
        self.contents = []
        dorgs = []

        for result in self.response_content["results"]:
            self.contents.append(result["document"]["structData"]["transcript"])
            self.uris.append(result["document"]["structData"]["uri"])
            dorgs.append(json.dumps(result["document"]["structData"]["organizations"][0]))

        # All organizations
        dorgs = [json.loads(o) for o in dorgs]

        # get org names
        org_names = set([org["name"] for org in dorgs])

        # get a list of organizations without duplicates
        for org_name in org_names:
            for dorg in dorgs:
                if dorg["name"] == org_name and dorg not in self.organizations:
                    self.organizations.append(dorg)

        # remove duplicates from these lists
        self.uris = list(set(self.uris))
        self.contents = list(set(self.contents))



In [5]:

q1 = "What are the responsibilities of the board members of a California community college?"
test = readVaiSearchResults(query=q1)





In [10]:
test.response_content["results"]

test.contents
test.organizations
test.uris

test.response_content




# test.organizations

{'results': [{'id': '19ab0e41059fa99cefd99646d4db331d',
   'document': {'name': 'projects/1062597788108/locations/global/collections/default_collection/dataStores/web-text-data-store_1750123044929/branches/0/documents/19ab0e41059fa99cefd99646d4db331d',
    'id': '19ab0e41059fa99cefd99646d4db331d',
    'structData': {'uri': 'https://www.cccco.edu/About-Us/Board-of-Governors#page-banner',
     'source_index': 'cccco',
     'available_time': '2025-05-01 18:34:28',
     'media_type': 'text',
     'country_origin': 'US',
     'organizations': [{'role': 'Informational site for all California community colleges.',
       'uri': 'https://www.cccco.edu',
       'name': 'California Community Colleges'}],
     'title': '',
     'content_index': 70,
     'transcript': "The California Community Colleges is guided by a process of participatory governance, and the Board of Governors of the California Community Colleges sets policy and provides guidance for the 73 districts and 116 colleges that const